# DAVID-Net Training — Kaggle

Crash-proof training with HuggingFace backup. Sessions can die — progress is safe on HF.

**Setup (one-time):**
1. Add `HF_TOKEN` to Kaggle Secrets
2. Attach all 8 datasets to `/kaggle/input/`
3. Select **T4 GPU** accelerator
4. Run All

In [ ]:
# Cell 1: GPU check
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Enable T4 in Settings.")

In [ ]:
# Cell 2: Clone repo
import os, sys
REPO = "/kaggle/working/david-net-av"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/MIHMahmudEli/david-net-av.git {REPO}
sys.path.insert(0, REPO)
print(f"Repo: {REPO}")

In [ ]:
# Cell 3: Load HF token from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF token loaded.")

In [ ]:
# Cell 4: Discover mounted datasets
from pathlib import Path
KAGGLE_INPUT = Path("/kaggle/input")

# Find all dataset folders (handles both direct and wrapper layouts)
mounted = {}
for d in sorted(KAGGLE_INPUT.iterdir()):
    if not d.is_dir(): continue
    subdirs = [s for s in d.iterdir() if s.is_dir()]
    if d.name == "datasets" and len(subdirs) > 1:
        for s in subdirs:
            mounted[s.name] = s
    else:
        mounted[d.name] = s if len(subdirs) == 1 and s.name.lower() == d.name else d

# Map to friendly names
DS_MAP = {
    "fakeavceleb-v1-2": "fakeavceleb",
    "lav-df": "lav-df",
    "dfdc-10": "dfdc-10",
    "deepfaketimit": "deepfaketimit",
    "celeb-df-v2": "celeb-df-v2",
    "asvpoof-2019-dataset-la": "asvpoof-2019",
    "in-the-wild-audio-deepfake": "in-the-wild",
    "wavefake": "wavefake",
}

datasets = {}
for key, friendly in DS_MAP.items():
    for name, path in mounted.items():
        if key in name.lower():
            datasets[friendly] = path
            print(f"  {friendly} -> {path}")
            break

print(f"\nFound {len(datasets)} datasets.")

In [ ]:
# Cell 5: Extract datasets (skips if already done)
import zipfile, tarfile

WORKING = Path("/kaggle/working")
DATA_DIR = WORKING / "data"
DATA_DIR.mkdir(exist_ok=True)

def extract_if_needed(name, src, dst):
    marker = dst / ".extracted"
    if marker.exists():
        print(f"  {name}: already done")
        return
    # Check if already loose files (no extraction needed)
    if any(src.rglob("*.mp4")) or any(src.rglob("*.wav")) or any(src.rglob("*.flac")):
        print(f"  {name}: loose files, linking")
        marker.touch()
        return
    # Extract zips
    for arch in src.rglob("*.zip"):
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with zipfile.ZipFile(arch) as zf:
                zf.extractall(dst)
            print("OK")
        except Exception as e:
            print(f"FAIL: {e}")
    marker.touch()

for name, path in datasets.items():
    dst = DATA_DIR / name
    dst.mkdir(exist_ok=True)
    extract_if_needed(name, path, dst)

print("Extraction done.")

In [ ]:
# Cell 6: Build manifests
MANIFEST_DIR = WORKING / "manifests"
MANIFEST_DIR.mkdir(exist_ok=True)
SPLIT_DIR = WORKING / "splits"
SPLIT_DIR.mkdir(exist_ok=True)

# FakeAVCeleb (existing converter)
fakeav_root = None
for candidate in [DATA_DIR / "fakeavceleb", DATA_DIR / "fakeavceleb" / "FakeAVCeleb_v1.2"]:
    if candidate.exists() and any((candidate / q).exists() for q in ["RealVideo-RealAudio", "FakeVideo-FakeAudio"]):
        fakeav_root = candidate
        break

if fakeav_root:
    !cd {REPO} && python scripts/build_manifest.py \
        --root {fakeav_root} \
        --out {MANIFEST_DIR}/fakeavceleb.jsonl \
        --splits-dir {SPLIT_DIR}/fakeavceleb --seed 42

# Other datasets
CONVERTERS = [
    ("dfdc-10", "dfdc-10"),
    ("deepfaketimit", "deepfaketimit"),
    ("celeb-df-v2", "celeb-df-v2"),
    ("asvpoof-2019", "asvpoof-2019"),
    ("in-the-wild", "in-the-wild"),
    ("wavefake", "wavefake"),
]

for ds_name, dataset_key in CONVERTERS:
    root = datasets.get(dataset_key) or DATA_DIR / ds_name
    if root.exists():
        !cd {REPO} && python scripts/build_manifests.py \
            --dataset {dataset_key} \
            --root {root} \
            --out {MANIFEST_DIR}/{ds_name}.jsonl \
            --splits-dir {SPLIT_DIR}/{ds_name}

# List what we have
print("\nManifests:")
for f in sorted(MANIFEST_DIR.glob("*.jsonl")):
    !wc -l {f}

In [ ]:
# Cell 7: Configure training
import yaml

CONFIG = {
    "run_id": "run_001",  # CHANGE for each new run
    "d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
    "use_sync": True, "use_disentangle": True, "compose_quadrant": False,
    "video_backbone": "fallback", "audio_backbone": "fallback",
    "video_model_name": "MCG-NJU/videomae-base", "audio_model_name": "microsoft/wavlm-base-plus",
    "freeze_blocks": 6, "freeze_feature_extractor": True,
    "init_from": None,
    "n_frames": 16, "audio_len": 64000, "shard_root": None, "feature_cache": None,
    "train_manifest": str(MANIFEST_DIR / "fakeavceleb.jsonl"),
    "modality_dropout": 0.15,
    "batch_size": 4, "num_workers": 2, "epochs": 30,
    "lr": 1e-4, "weight_decay": 1e-4, "log_every": 10,
    "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
    "loss_weights": {"v": 1.0, "a": 1.0, "quad": 0.5, "loc": 0.5, "sync": 0.3, "disentangle": 0.1},
    "seed": 42,
}

config_path = WORKING / "train_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(CONFIG, f)
print(f"Run: {CONFIG['run_id']}")
print(f"Manifest: {CONFIG['train_manifest']}")

In [ ]:
# Cell 8: Train!
!cd {REPO} && python -m src.training.train \
    --config {config_path} \
    --run-id {CONFIG['run_id']}

In [ ]:
# Cell 9: Verify HF backup
from huggingface_hub import HfApi
api = HfApi(token=hf_token)
try:
    files = list(api.list_repo_tree("MoshinAli/david-net-av-backup",
                                     path_in_repo=f"runs/{CONFIG['run_id']}",
                                     repo_type="model", recursive=True))
    print(f"Files on HF for {CONFIG['run_id']}:")
    for f in files:
        if hasattr(f, 'path'): print(f"  {f.path}")
except Exception as e:
    print(f"Could not list HF repo: {e}")